# 🏥 Phase 3.1 — Clinical-Grade Active-Ledger Evaluation (Kaggle)

> **FYP Project**: Resilient Clinical Federated Learning via Active-Ledger and Latent Diffusion

This notebook executes the **flagship 40-round clinical simulation** on a Kaggle GPU to prove clinical viability (F1 ≥ 0.75).

## What This Notebook Proves for Your FYP

| FYP Requirement | How This Notebook Proves It |
|:---|:---|
| **Clinical F1 ≥ 0.75** | Runs 40 FL rounds with full LDM augmentation → per-class F1 bar charts |
| **Active-Ledger PoC Filtering** | Deploys `FLLogger.sol` on Ganache, computes PoC scores, Top-7 selection |
| **Ledger-Guided Diffusion** | Clients detect minority classes, request blockchain permits, generate 500 synthetic ECGs/class |
| **Byzantine Resilience** | 2 of 10 clients submit Gaussian noise; PoC scoring filters them out |
| **Reproducibility** | Fixed seeds (42), deterministic Ganache, one-click execution |

## Pipeline
```
Install Deps → Start Ganache → Deploy FLLogger.sol → Download MIT-BIH
→ Preprocess → Partition (Non-IID) → Quick-Test (5 rounds) → Full Run (40 rounds)
→ Save Results to Kaggle Output
```

---

> ⚠️ **IMPORTANT**: Select **GPU T4 ×2** (or P100) in Settings → Accelerator before running!

In [ ]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================

# CRITICAL: Kaggle has PyTorch pre-installed with the correct CUDA
# kernels for their GPUs. We must NOT let pip upgrade torch, or
# CUDA will break with 'no kernel image available' errors.

import torch
print(f'Kaggle PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Step 1: Install packages that do NOT depend on torch
!pip install -q wfdb flwr web3 py-solc-x pyyaml

# Step 2: Install diffusers & accelerate WITHOUT touching torch
!pip install -q --no-deps diffusers accelerate

# Step 3: Install diffusers' non-torch dependencies
!pip install -q safetensors huggingface-hub filelock regex requests importlib-metadata

# Step 4: Install the Solidity compiler
from solcx import install_solc
install_solc('0.8.19')
print('\u2705 Solidity compiler 0.8.19 installed')

# Step 5: Install Ganache (local Ethereum blockchain)
!npm install -g ganache 2>/dev/null && echo '\u2705 Ganache installed' || echo '\u26a0\ufe0f Retrying...'

# Step 6: Verify GPU and check compute capability
print(f'\n{"="*60}')
GPU_OK = False
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cc = torch.cuda.get_device_capability(0)
    print(f'GPU: {gpu_name}')
    print(f'VRAM: {vram:.1f} GB')
    print(f'Compute Capability: {cc[0]}.{cc[1]}')
    if cc >= (7, 0):
        print(f'\u2705 GPU fully supported! Everything will run on CUDA.')
        GPU_OK = True
    else:
        print(f'\u26a0\ufe0f GPU compute capability {cc[0]}.{cc[1]} is too old!')
        print(f'   P100 (6.0) is not supported by this PyTorch build.')
        print(f'   \u2192 Go to Settings \u2192 Accelerator \u2192 select GPU T4 \u00d72')
        print(f'   Diffusion will fall back to CPU if you continue.')
else:
    print('\u26a0\ufe0f  NO GPU DETECTED')
    print('   Go to Settings \u2192 Accelerator \u2192 GPU T4 \u00d72')
print(f'{"="*60}')
print('\n\u2705 Environment setup complete!')

In [ ]:
# ============================================================
# Cell 2: Copy Repository Code to Working Directory
# ============================================================
# Your repo must be uploaded as a Kaggle Dataset.
# Kaggle often nests files in subdirectories, so we auto-detect
# the actual repo root by searching for main.py.

import shutil, os, glob

DATASET_PATHS = glob.glob('/kaggle/input/*')
print(f'Available datasets: {DATASET_PATHS}')

# Auto-detect the dataset
dataset_path = None
for p in DATASET_PATHS:
    name = os.path.basename(p).lower()
    if 'fyp' in name or 'blockchain' in name or 'fl' in name:
        dataset_path = p
        break

if dataset_path is None and len(DATASET_PATHS) >= 1:
    dataset_path = DATASET_PATHS[0]

if dataset_path is None:
    raise FileNotFoundError(
        '\u274c Could not find your dataset!\n'
        'Click "Add Data" in the right panel \u2192 search your dataset \u2192 Add it'
    )

print(f'Dataset found: {dataset_path}')
print(f'Contents: {os.listdir(dataset_path)}')

# --- Find the ACTUAL repo root (handles Kaggle nesting) ---
# Kaggle often places files inside an extra subdirectory.
# We search for main.py to locate the true repo root.
repo_root = None

# Check 1: main.py directly in dataset_path
if os.path.isfile(os.path.join(dataset_path, 'main.py')):
    repo_root = dataset_path
else:
    # Check 2: main.py one level deeper
    for item in os.listdir(dataset_path):
        candidate = os.path.join(dataset_path, item)
        if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, 'main.py')):
            repo_root = candidate
            break

if repo_root is None:
    # Check 3: search recursively (up to 3 levels deep)
    for root, dirs, files in os.walk(dataset_path):
        if root.count(os.sep) - dataset_path.count(os.sep) > 3:
            break
        if 'main.py' in files and 'core' in dirs:
            repo_root = root
            break

if repo_root is None:
    raise FileNotFoundError(
        '\u274c Could not find main.py in your dataset!\n'
        'Make sure you uploaded the FYP-Blockchain-FL folder correctly.\n'
        f'Dataset contents: {os.listdir(dataset_path)}'
    )

print(f'\n\u2705 Repo root found: {repo_root}')
print(f'   Contents: {sorted(os.listdir(repo_root))}')

# --- Copy repo to /kaggle/working/ ---
WORK_DIR = '/kaggle/working'

for item in os.listdir(repo_root):
    if item.startswith('.'): continue
    src = os.path.join(repo_root, item)
    dst = os.path.join(WORK_DIR, item)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'  \U0001f4c1 Copied: {item}/')
    else:
        shutil.copy2(src, dst)
        print(f'  \U0001f4c4 Copied: {item}')

os.chdir(WORK_DIR)

# Verify critical files exist
critical = ['main.py', 'config.yaml', 'core', 'benchmarks', 'contracts']
missing = [f for f in critical if not os.path.exists(f)]
if missing:
    print(f'\u274c Missing critical files: {missing}')
    print(f'   Current dir contents: {os.listdir(".")}')
else:
    print(f'\n\u2705 All critical files present!')
    print(f'   Working directory: {os.getcwd()}')
    print(f'   Files: {sorted([f for f in os.listdir(".") if not f.startswith(".")])}')

In [ ]:
# ============================================================
# Cell 3: Start Blockchain Infrastructure
# ============================================================
import subprocess, time

print('[..] Starting Ganache local blockchain...')
ganache_proc = subprocess.Popen(
    'NODE_OPTIONS=--max-old-space-size=8192 ganache -p 8545 --accounts 15 --deterministic --gasLimit 30000000',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(10)  # Wait for Ganache to fully initialize

if ganache_proc.poll() is not None:
    print('[..] Retrying with ganache-cli...')
    ganache_proc = subprocess.Popen(
        'NODE_OPTIONS=--max-old-space-size=8192 ganache-cli -p 8545 --accounts 15 --deterministic --gasLimit 30000000',
        shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    time.sleep(10)

if ganache_proc.poll() is None:
    print(f'\u2705 Ganache running (PID: {ganache_proc.pid})')
else:
    print('\u274c Ganache failed to start! Check npm installation.')

# Deploy the FLLogger smart contract
print('\n[..] Deploying FLLogger smart contract...')
!PYTHONPATH=/kaggle/working python core/deploy_contract.py
print('\n\u2705 Blockchain infrastructure ready!')

In [ ]:
# ============================================================
# Cell 4: Data Preparation (MIT-BIH Arrhythmia Dataset)
# ============================================================
import os

# Set PYTHONPATH so scripts can find 'core' package
os.environ['PYTHONPATH'] = '/kaggle/working'

print('=' * 60)
print('STEP 1/3: Downloading MIT-BIH Arrhythmia records (~120 MB)...')
print('=' * 60)
!rm -rf data/* checkpoints/*
!PYTHONPATH=/kaggle/working python core/download_data.py

print('\n' + '=' * 60)
print('STEP 2/3: Preprocessing ECG signals (segmentation + features)...')
print('=' * 60)
!PYTHONPATH=/kaggle/working python core/preprocess_data.py

print('\n' + '=' * 60)
print('STEP 3/3: Partitioning data across 10 clients (Non-IID Dirichlet)...')
print('=' * 60)
!PYTHONPATH=/kaggle/working python core/partition_data.py

# Verify
pdir = 'data/partitioned'
if os.path.exists(pdir):
    files = os.listdir(pdir)
    print(f'\n\u2705 Data preparation complete! {len(files)} partition files created.')
else:
    print('\u274c Data preparation may have failed \u2014 check errors above')

---

## ⚡ Quick-Test Sanity Check

The next cell runs a **reduced configuration** (5 rounds, 5 diffusion steps, 50 samples) to verify the entire pipeline works before the full 6-10 hour run.

**Expected time**: ~15-25 minutes on a T4 GPU.

If this completes and shows F1 scores, the full pipeline is verified.

In [ ]:
# ============================================================
# Cell 6: Quick-Test (5 Rounds) — Sanity Check
# ============================================================
import sys, torch
sys.path.insert(0, '/kaggle/working')

# Force clean import
for mod_name in list(sys.modules.keys()):
    if 'benchmarks' in mod_name or 'core' in mod_name:
        del sys.modules[mod_name]

# Auto-detect GPU capability
import core.diffusion as diff_module
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability(0)
    if cc < (7, 0):
        # P100 or older: patch diffusion to CPU
        _orig_init = diff_module.ECGDiffusionGenerator.__init__
        def _cpu_init(self):
            _orig_init(self)
            self.device = 'cpu'
            self.unet = self.unet.to('cpu')
        diff_module.ECGDiffusionGenerator.__init__ = _cpu_init
        print(f'\u26a0\ufe0f GPU cc={cc[0]}.{cc[1]} too old \u2014 diffusion on CPU, rest on CUDA')
    else:
        print(f'\u2705 GPU cc={cc[0]}.{cc[1]} \u2014 everything on CUDA!')

import benchmarks.run_final_clinical_evaluation as clinical_module

# Patch constants for quick test
clinical_module.NUM_ROUNDS = 5
clinical_module.DIFFUSION_STEPS = 5
clinical_module.SYNTHETIC_QUANTITY = 50
clinical_module.CHECKPOINT_EVERY = 5

print('=' * 70)
print('\u26a1 QUICK-TEST MODE')
print(f'   Rounds          : {clinical_module.NUM_ROUNDS}')
print(f'   Diffusion Steps : {clinical_module.DIFFUSION_STEPS}')
print(f'   Synthetic Qty   : {clinical_module.SYNTHETIC_QUANTITY}')
print('=' * 70)

try:
    clinical_module.main()
    print('\n' + '=' * 70)
    print('\u2705 QUICK-TEST PASSED!')
    print('   Proceed to the next cells for the full 40-round run.')
    print('=' * 70)
except Exception as e:
    print(f'\n\u274c QUICK-TEST FAILED: {e}')
    import traceback; traceback.print_exc()

---

## 🚀 Full Clinical Evaluation (40 Rounds)

The next cells restart the blockchain for a **clean state** and run the full evaluation.

**Expected time**: ~9-10 hours on a T4 GPU.

> 💡 **Tip**: You can close the browser tab — Kaggle continues running in the background.

### What happens during the full run:
1. **40 FL rounds** with 8 honest + 2 Byzantine clients
2. Each round: local training → on-chain logging → PoC scoring → Top-7 selection
3. Trusted clients (PoC > 0.4) get **diffusion permits** from the blockchain
4. **500 synthetic ECGs** per minority class via 1D-UNet LDM (50 inference steps)
5. Checkpoints saved every 10 rounds + final model at round 40

In [ ]:
# ============================================================
# Cell 8: Restart Blockchain for Clean State
# ============================================================
import subprocess, time, os, shutil

# Kill existing Ganache
print('[..] Stopping previous Ganache...')
try:
    ganache_proc.terminate()
    ganache_proc.wait(timeout=5)
except:
    pass
os.system('pkill -f ganache 2>/dev/null || true')
time.sleep(3)

# Start fresh Ganache
print('[..] Starting fresh Ganache...')
ganache_proc = subprocess.Popen(
    'NODE_OPTIONS=--max-old-space-size=8192 ganache -p 8545 --accounts 15 --deterministic --gasLimit 30000000',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(10)

if ganache_proc.poll() is None:
    print(f'\u2705 Fresh Ganache running (PID: {ganache_proc.pid})')
else:
    ganache_proc = subprocess.Popen(
        'NODE_OPTIONS=--max-old-space-size=8192 ganache-cli -p 8545 --accounts 15 --deterministic --gasLimit 30000000',
        shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    time.sleep(10)
    print(f'\u2705 Fresh Ganache-CLI running (PID: {ganache_proc.pid})')

# Redeploy contract
print('\n[..] Redeploying FLLogger smart contract...')
!python core/deploy_contract.py

# Clean quick-test artifacts
for path in ['checkpoints', 'final_clinical_f1_scores.pdf']:
    if os.path.isdir(path):
        shutil.rmtree(path)
    elif os.path.isfile(path):
        os.remove(path)
print('\n\u2705 Clean blockchain state. Ready for full 40-round run!')

In [ ]:
# ============================================================
# Cell 9: Full Clinical Compute — 40 Rounds
# ============================================================
# Auto-detects GPU capability:
#   T4+ (cc>=7.0): everything on CUDA (fastest)
#   P100 (cc<7.0): diffusion on CPU, rest on CUDA
#
# Configuration (from benchmarks/run_final_clinical_evaluation.py):
#   NUM_ROUNDS=40, DIFFUSION_STEPS=50, SYNTHETIC_QUANTITY=500
#   8 honest + 2 Byzantine clients, Top-K=7
#
# Estimated time: ~9-10 hours on T4 GPU
# ============================================================
import torch

need_patch = False
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability(0)
    need_patch = cc < (7, 0)

if need_patch:
    # Old GPU: write wrapper that patches diffusion to CPU
    wrapper = '''
import sys
sys.path.insert(0, '/kaggle/working')
import torch, core.diffusion as diff_module
_orig = diff_module.ECGDiffusionGenerator.__init__
def _p(self):
    _orig(self)
    self.device = 'cpu'
    self.unet = self.unet.to('cpu')
diff_module.ECGDiffusionGenerator.__init__ = _p
cc = torch.cuda.get_device_capability(0)
print(f'[AUTO] GPU cc={cc[0]}.{cc[1]} - diffusion on CPU, training on CUDA')
from benchmarks.run_final_clinical_evaluation import main
main()
'''
    with open('run_clinical_gpu_safe.py', 'w') as f:
        f.write(wrapper)
    print(f'\u26a0\ufe0f GPU cc={cc[0]}.{cc[1]} - using CPU fallback for diffusion')
    !PYTHONPATH=/kaggle/working python run_clinical_gpu_safe.py
else:
    # T4 or newer: everything on GPU!
    if torch.cuda.is_available():
        print(f'\u2705 GPU cc={cc[0]}.{cc[1]} - full CUDA mode!')
    !PYTHONPATH=/kaggle/working python main.py --mode clinical-compute

In [ ]:
# ============================================================
# Cell 10: Collect & Display Results
# ============================================================
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('=' * 70)
print('\U0001f4ca PHASE 3.1 \u2014 FINAL RESULTS')
print('=' * 70)

class_names = ['Normal', 'LBBB', 'RBBB', 'APB', 'PVC']

try:
    f1_scores = np.load('checkpoints/final_f1_scores.npy')
    round_f1  = np.load('checkpoints/round_mean_f1.npy')

    print('\n\U0001f4c8 Per-Class F1 Scores (Final Round):')
    print('-' * 40)
    for name, score in zip(class_names, f1_scores):
        status = '\u2705' if score >= 0.75 else '\u26a0\ufe0f'
        print(f'  {status} {name:8s}: {score:.4f}')

    mean_f1 = np.mean(f1_scores)
    print(f'\n  {"="*30}')
    tag = '\u2705 CLINICAL VIABILITY ACHIEVED' if mean_f1 >= 0.75 else '\u26a0\ufe0f Below threshold'
    print(f'  Mean F1: {mean_f1:.4f} \u2014 {tag}')
    print(f'  {"="*30}')

    print(f'\n\U0001f4c9 Training Curve:')
    for r in [1, 10, 20, 40]:
        idx = min(r-1, len(round_f1)-1)
        print(f'  Round {r:2d}: {round_f1[idx]:.4f}')

    # Inline plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    colors = ['#2C7BB6', '#D7191C', '#1A9641', '#FDAE61', '#762A83']
    bars = ax1.bar(class_names, f1_scores, color=colors, edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, f1_scores):
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax1.set_ylim(0, 1.1)
    ax1.set_ylabel('F1 Score')
    ax1.set_title('Final Per-Class F1 Scores\n(40 Rounds, Ledger-Guided Diffusion)')
    ax1.axhline(y=0.75, color='red', linestyle='--', alpha=0.5, label='Clinical Threshold')
    ax1.legend()
    ax1.grid(axis='y', linestyle='--', alpha=0.4)

    ax2.plot(range(1, len(round_f1)+1), round_f1, color='#2C7BB6', linewidth=2, marker='o', markersize=3)
    ax2.set_xlabel('Round')
    ax2.set_ylabel('Mean F1 Score')
    ax2.set_title('Training Curve \u2014 Mean F1 vs Round')
    ax2.axhline(y=0.75, color='red', linestyle='--', alpha=0.5, label='Clinical Threshold')
    ax2.legend()
    ax2.grid(linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.savefig('results_inline.png', dpi=150, bbox_inches='tight')
    plt.show()

except FileNotFoundError:
    print('\u26a0\ufe0f Results not found. The full run may not have completed.')

# List output files
print('\n' + '=' * 70)
print('\U0001f4c1 OUTPUT FILES (download from Kaggle Output tab):')
print('=' * 70)
for root, dirs, files in os.walk('/kaggle/working'):
    for f in sorted(files):
        if f.endswith(('.pdf', '.npy', '.pth', '.png')):
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / 1e6
            rel = os.path.relpath(fpath, '/kaggle/working')
            print(f'  {rel}: {size_mb:.2f} MB')

print('\n\u2705 All results saved. Go to the Output tab to download them.')

---

## 📊 Understanding Your Results

### Per-Class F1 Scores
| Class | Name | Description |
|:-----:|:-----|:------------|
| 0 | Normal | Normal sinus rhythm |
| 1 | LBBB | Left Bundle Branch Block |
| 2 | RBBB | Right Bundle Branch Block |
| 3 | APB | Atrial Premature Beat |
| 4 | PVC | Premature Ventricular Contraction |

- **F1 ≥ 0.75** → Clinical viability achieved ✅
- Classes 3 (APB) and 4 (PVC) are minority classes that benefit most from Ledger-Guided Diffusion.

### Training Curve
- **Rounds 1-10**: Rapid improvement as the model learns basic ECG patterns
- **Rounds 10-30**: Diffusion augmentation boosts minority class F1
- **Rounds 30-50**: Convergence at clinical-grade performance
- Byzantine clients are consistently filtered out by PoC scoring

### FYP Defense Talking Points
1. **"Why not just FedAvg?"** → Without PoC filtering, Byzantine clients corrupt the model
2. **"Why not blind diffusion?"** → Without blockchain trust verification, augmentation is uncontrolled
3. **"Is it clinically viable?"** → F1 ≥ 0.75 on MIT-BIH proves clinical-grade performance
4. **"Is it reproducible?"** → Fixed seeds + blockchain audit trail = full reproducibility

### Files for Your FYP Report
- `final_clinical_f1_scores.pdf` → Figure in Results chapter
- `checkpoints/final_f1_scores.npy` → Raw data for tables
- `checkpoints/round_mean_f1.npy` → Training curve data
- `checkpoints/global_model_round_50.pth` → Trained model for further experiments